# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
import numpy as np, pandas as pd

# --- Bootstrap: locate the repo root so this runs from anywhere (local or Colab) ---
CSV = "data/raw/content_refresh_anonymized.csv"
REPO_URL = "https://github.com/thany-8/content-refresh-prioritizer"
REPO_DIR = "content-refresh-prioritizer"

def find_root(start, marker=CSV, up=6):
    """Walk up from `start` until a dir containing `marker` is found (repo root)."""
    d = os.path.abspath(start)
    for _ in range(up + 1):
        if os.path.exists(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    return None

root = find_root(os.getcwd())
if root is None:  # e.g. a fresh Colab VM: clone the public repo, then use it
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    root = os.path.abspath(REPO_DIR)
os.chdir(root)
sys.path.insert(0, os.path.join(root, "scripts"))   # reuse the pristine pipeline, don't re-edit it

assert os.path.exists(CSV), f"starter CSV not found under {root}"
df = pd.read_csv(CSV)
print("loaded", df.shape[0], "rows x", df.shape[1], "cols")

loaded 30000 rows x 44 cols


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the vector by **reusing the pristine pipeline's honest recipe** (`scripts/01_prepare_features.py` + the `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` lists in `scripts/ml_utils.py`) rather than re-inventing it — same filters, same label, same engineered columns:

- **Filter** to `impressions_90d > 0` and `content_age_days >= 90`, then de-dupe on `content_id`.
- **Label** `is_declining_label = (trend_direction == "down")` — a defined proxy, never a feature.
- **Engineer** `log1p` of the heavy-tailed traffic totals (`impressions/clicks/sessions/ai_sessions`).
- **Categoricals** → `"unknown"` category, then one-hot.
- **Missing numerics** → 0, **but** because missingness follows `content_type` (feedly = 100% no keyword data; keyword = ~28% no `word_count`), I add `has_word_count` / `has_keyword_data` flags so a blind `fillna(0)` can't silently encode content type into the features.

In [2]:
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# Mirror the pristine pipeline's honest recipe (scripts/01_prepare_features.py).
work = (df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
        .drop_duplicates("content_id").reset_index(drop=True))
y = work["trend_direction"].str.lower().eq("down").astype(int)   # is_declining_label (proxy target)
groups = work["client_id"]                                       # grouped-split key: context, NOT a feature

# Engineered numerics: log1p the heavy-tailed traffic totals (keep the pipeline's names).
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    work["log_" + c] = np.log1p(pd.to_numeric(work[c], errors="coerce").fillna(0))

# Missingness follows content_type -> has_-flags so fillna(0) can't smuggle a category signal.
work["has_word_count"]  = work["word_count"].notna().astype(int)
work["has_keyword_data"] = work["search_volume"].notna().astype(int)
extra_flags = ["has_word_count", "has_keyword_data"]

numeric = work[MODEL_NUMERIC_FEATURES + extra_flags].apply(pd.to_numeric, errors="coerce").fillna(0)
categorical = pd.get_dummies(work[MODEL_CATEGORICAL_FEATURES].astype(str).fillna("unknown"))
X = pd.concat([numeric, categorical], axis=1)

print("feature matrix:", X.shape, "| base rate (down):", round(y.mean() * 100, 1), "%")
print("numeric features:", len(MODEL_NUMERIC_FEATURES) + len(extra_flags),
      "| one-hot categorical columns:", categorical.shape[1])

feature matrix: (30000, 54) | base rate (down): 54.2 %
numeric features: 20 | one-hot categorical columns: 34


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The model uses **18 numeric + 8 categorical** source fields (plus 2 `has_`-flags). Meaning comes from `docs/data-dictionary.md`; the table below records the two things that decide honesty — **how missing values are filled** and **when each field is knowable**.

The `available_when` answer is subtle on this slice: every feature is a **content descriptor** (keyword context, length, age) or a **90-day-window aggregate**, all knowable at snapshot time — but the 90d aggregates *overlap the label's last-30-day window* (flagged below). On a single snapshot there is no fully clean pre-label window; the honest fix is the Week-3 warehouse's forward-label design. I keep the 90d aggregates but flag the residual overlap rather than pretend it away.

In [3]:
# For each MODEL feature source: fraction missing BEFORE fill, fill rule, and when it's knowable.
def pct_missing(col):
    return round(work[col].isna().mean() * 100, 1) if col in work.columns else float("nan")

src = {"log_impressions_90d": "impressions_90d", "log_clicks_90d": "clicks_90d",
       "log_sessions_90d": "sessions_90d", "log_ai_sessions_90d": "ai_sessions_90d"}
overlap = {"log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "ctr", "avg_position",
           "engagement_rate", "scroll_rate", "ai_traffic_pct"}

notes = pd.DataFrame({
    "feature": MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES,
    "kind": ["numeric"] * len(MODEL_NUMERIC_FEATURES) + ["categorical"] * len(MODEL_CATEGORICAL_FEATURES),
})
notes["source_%missing"] = notes["feature"].map(lambda f: pct_missing(src.get(f, f)))
notes["fill"] = notes["kind"].map({"numeric": "0 (+has_-flag if patterned)", "categorical": "'unknown'"})
notes["overlaps_label_30d"] = notes["feature"].isin(overlap)
print(notes.to_string(index=False))

               feature        kind  source_%missing                        fill  overlaps_label_30d
         search_volume     numeric              8.2 0 (+has_-flag if patterned)               False
           competition     numeric              8.2 0 (+has_-flag if patterned)               False
                   cpc     numeric              8.2 0 (+has_-flag if patterned)               False
            word_count     numeric             25.7 0 (+has_-flag if patterned)               False
            char_count     numeric             25.7 0 (+has_-flag if patterned)               False
   log_impressions_90d     numeric              0.0 0 (+has_-flag if patterned)                True
        log_clicks_90d     numeric              0.0 0 (+has_-flag if patterned)                True
      log_sessions_90d     numeric              0.0 0 (+has_-flag if patterned)                True
   log_ai_sessions_90d     numeric              0.0 0 (+has_-flag if patterned)                True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I attack my own model before trusting any number. Three checks from the taxonomy:

1. **Base rate first.** The label is 54.2% `down`, so accuracy is meaningless — I judge ranking with ROC-AUC and precision@K *next to* that base rate.
2. **Honest split vs random split.** Grouped by `client_id` (client-holdout) the model scores **ROC-AUC ≈ 0.69**; a random split inflates it to **≈ 0.78**. That **gap is client memorization** — a random split would have let me fake ~9 points of skill. Top-of-queue precision@100 ≈ **85%** vs the 54.2% base is the honest, useful signal.
3. **Deliberately add leaks and watch the score confess.** Adding the label-derived `trend_pct` sends grouped AUC to **1.0**; adding the overlapping-window 30-day impressions sends it to **0.999** (it reconstructs the last-vs-prev ratio the label is built from). Both jumps prove the harness works — then I remove them and keep the honest 0.69.

**Residual I can't remove here:** even the honest features (90d aggregates) overlap the label's 30d window, so 0.69 is *honest-ish*, not provably clean — the fully clean design is the warehouse forward label.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
from ml_utils import precision_at_k

BASE = y.mean() * 100
gkf = GroupKFold(n_splits=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def oof_proba(features, cv, groups=None):
    """Out-of-fold P(down) — never in-sample."""
    model = HistGradientBoostingClassifier(random_state=0)
    return cross_val_predict(model, features, y, cv=cv, groups=groups, method="predict_proba")[:, 1]

# Honest model: grouped (client-holdout) vs random split -> the GAP is memorization.
p_group = oof_proba(X, gkf, groups)
p_rand  = oof_proba(X, skf)
print(f"base rate (majority = 'down'): {BASE:.1f}%")
print(f"HONEST grouped-CV ROC-AUC: {roc_auc_score(y, p_group):.3f}")
print(f"HONEST random-CV  ROC-AUC: {roc_auc_score(y, p_rand):.3f}   (gap = client memorization)")
for K in (100, 500, 1000):
    print(f"  honest precision@{K}: {precision_at_k(y, p_group, K) * 100:.1f}%   (vs {BASE:.1f}% base)")

# Attack: add leaky columns, watch AUC jump toward 1.0 (proves the test harness itself works).
def with_cols(cols):
    Xa = X.copy()
    for c in cols:
        Xa[c] = pd.to_numeric(work[c], errors="coerce").fillna(0)
    return Xa

auc_leak_label  = roc_auc_score(y, oof_proba(with_cols(["trend_pct"]), gkf, groups))
auc_leak_window = roc_auc_score(y, oof_proba(with_cols(["impressions_last_30d", "impressions_prev_30d"]), gkf, groups))
print(f"\nLEAK #1 label-derived (+trend_pct):            AUC {auc_leak_label:.3f}  <- the confession")
print(f"LEAK #2 overlapping window (+30d impressions): AUC {auc_leak_window:.3f}  <- reconstructs the ratio")

base rate (majority = 'down'): 54.2%
HONEST grouped-CV ROC-AUC: 0.690
HONEST random-CV  ROC-AUC: 0.781   (gap = client memorization)
  honest precision@100: 85.0%   (vs 54.2% base)
  honest precision@500: 78.2%   (vs 54.2% base)
  honest precision@1000: 75.4%   (vs 54.2% base)



LEAK #1 label-derived (+trend_pct):            AUC 1.000  <- the confession
LEAK #2 overlapping window (+30d impressions): AUC 0.999  <- reconstructs the ratio


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Every refusal maps to the leakage taxonomy:

| Field(s) | Bucket | Why refused |
|---|---|---|
| `trend_direction`, `trend_pct` | label-derived | the label *is* `trend_direction == 'down'`; `trend_pct` defines it |
| `impressions_last_30d`, `impressions_prev_30d` | label-derived | the trend's raw inputs — leak #1/#2 above |
| `clicks_last_30d/prev_30d`, `sessions_last_30d/prev_30d` | overlapping window | sit inside the label's 30d window, track the same dynamics |
| `provider_used`, `model_used` | decision-derived | how the article was generated (product/pipeline metadata), not a content signal |
| `content_id`, `client_id` | context | pseudonymous IDs — grouped splits only, never learned from |

**Privacy:** IDs stay pseudonymous and are used only to *group* the split; no client names, URLs, or raw queries enter the features or the outputs. The results are **decision-support / directional**, not causal.

In [5]:
excluded = {
    "trend_direction / trend_pct": "label source (is_declining_label == trend_direction=='down') — label-derived",
    "impressions_last_30d / impressions_prev_30d": "the trend's raw inputs — reconstruct the label",
    "clicks_last_30d/prev_30d, sessions_last_30d/prev_30d": "inside the label's 30d window — overlapping-window leak",
    "provider_used / model_used": "generation / product metadata — decision-derived, not a content signal",
    "content_id / client_id": "pseudonymous IDs — context for grouped splits, never a feature",
}
for field, why in excluded.items():
    print(f"- {field}: {why}")

print("\nResidual (unfixable on this slice): the 90d aggregates I DO use overlap the label's last-30d")
print("window -> the honest AUC is 'honest-ish'; a fully clean feature window needs the Week-3 warehouse.")

- trend_direction / trend_pct: label source (is_declining_label == trend_direction=='down') — label-derived
- impressions_last_30d / impressions_prev_30d: the trend's raw inputs — reconstruct the label
- clicks_last_30d/prev_30d, sessions_last_30d/prev_30d: inside the label's 30d window — overlapping-window leak
- provider_used / model_used: generation / product metadata — decision-derived, not a content signal
- content_id / client_id: pseudonymous IDs — context for grouped splits, never a feature

Residual (unfixable on this slice): the 90d aggregates I DO use overlap the label's last-30d
window -> the honest AUC is 'honest-ish'; a fully clean feature window needs the Week-3 warehouse.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.